## END TO END PROJECT USING SIMPLE RNN 

In [ ]:
# 1. Import all needed libraries
# (TensorFlow, IMDB dataset, padding, RNN layers)
# 2. Load the IMDB movie review dataset (already converted to numbers)
# 3. Prepare word-index and reverse-word-index (to understand reviews)
# 4. Pad sequences so all reviews have the same length (Required for RNN)
# 5. Build the model:
#    - Embedding: turn word IDs into vectors
#    - SimpleRNN: learn the sequence pattern
#    - Dense: output 0/1 sentiment
# 6. Compile the model with optimizer + loss function
# 7. Add EarlyStopping to stop training when no improvement
# 8. Train the model on the training data
# 9. Evaluate the model on test data

In [17]:
import numpy as np
# NumPy: fast math on arrays (for data handling, reshaping, etc.)

import tensorflow as tf
# TensorFlow: main deep learning framework.

from tensorflow.keras.datasets import imdb
# imdb: built-in movie review dataset (text + sentiment labels).

from tensorflow.keras.preprocessing import sequence
# sequence: tools like pad_sequences to make all sequences same length.

from tensorflow.keras.models import Sequential
# Sequential: lets us build the model layer-by-layer.

from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
# Embedding: turns word indices into dense vectors.
# SimpleRNN: basic recurrent layer for sequence learning.
# Dense: fully connected layer for final prediction (e.g., sentiment).

In [18]:
## Load the IMDb dataset
# max_features = 10000 → keep only the top 10k most frequent words.
# imdb.load_data(...) returns encoded reviews:
#   X_train, X_test = sequences of word indices
#   y_train, y_test = sentiment labels (0 = negative, 1 = positive)

max_features=10000 ##vocabulary size
(X_train,y_train),(X_test,y_test)=imdb.load_data(num_words=max_features)

# Printing shapes shows:
# - How many reviews we have
# - One label per review
print(f'Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {X_train.shape}, Testing labels shape: {y_test.shape}')


Training data shape: (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)


In [ ]:
## Inspect a sample review and its label
sample_review=X_train[0]
sample_label=y_train[0]

print(f"Sample review (as integers):{sample_review}")
print(f'Sample label: {sample_label}')


In [ ]:
### Mapping word index back to words (for understanding)

"""word_index.items() gives pairs like:
("the", 1), ("movie", 2), ("good", 3), ...

for key, value in word_index.items()
loops through each pair:
key = "the", value = 1

{value: key ...}
creates a new dictionary where the index becomes the key
and the word becomes the value:""" 

word_index=imdb.get_word_index()

reverse_word_index = {value: key for key,value in word_index.items()}
reverse_word_index

In [ ]:
decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in sample_review])
decoded_review 

"""for i in sample_review
Loops through every number in the encoded review.
Example:
sample_review = [1, 14, 20, 5, ...]

i - 3

IMDB reserves 0, 1, 2 for special tokens.
So real words start from index 3.
We subtract 3 to match our reverse index.

-> reverse_word_index.get(i - 3, '?')
Looks up the word for that index.
If found → return the word
If not found → return '?'

-> [...]
Creates a list of decoded words.
' '.join(...)
Joins the list into a proper sentence with spaces."""

In [19]:
from tensorflow.keras.preprocessing import sequence
max_len = 500
X_train= sequence.pad_sequences(X_train,maxlen=max_len)
X_test= sequence.pad_sequences(X_test,maxlen=max_len)

In [ ]:
X_train[0]

In [20]:
## Train Simple RNN
model=Sequential()

# Embedding layer: converts word indices → 128-dim dense vectors
model.add(Embedding(max_features, 128, input_length=max_len))

# SimpleRNN layer: processes the sequence step-by-step and learns patterns
model.add(SimpleRNN(128, activation='relu'))

# Output layer: sigmoid gives a probability (positive vs negative review)
model.add(Dense(1, activation="sigmoid"))

d:\AI\KrishNaik_Academy\Coding\NLP\ANN\venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
# Compile the model
# optimizer='adam' → helps the model learn efficiently
# loss='binary_crossentropy' → best for 0/1 sentiment classification
model.compile(optimizer='adam', loss='binary_crossentropy')

In [22]:
## Create an instance of Early stopping callback 

from tensorflow.keras.callbacks import EarlyStopping
earlystopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

# EarlyStopping:
# Stops training early when the model stops improving.
# monitor='val_loss' → watches validation loss.
# patience=5 → waits 5 epochs before stopping.
# restore_best_weights=True → keeps the best version of the model.

In [25]:
# Train the model
# X_train, y_train → training data
# epochs=10 → model will train for max 10 passes over the data
# batch_size=32 → After seeing 32 samples, it updates its weights once.
# validation_split=0.2 → 20% of training data used for validation
# callbacks=[earlystopping] → stop early if validation loss stops improvin
history=model.fit(
    X_train,y_train,epochs=10,batch_size=32,
    validation_split=0.2,
    callbacks=[earlystopping]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 99s 155ms/step - loss: 3018.7261 - val_loss: 0.6390
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 99s 158ms/step - loss: 0.5774 - val_loss: 0.5731
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 97s 156ms/step - loss: 0.4462 - val_loss: 0.5275
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 95s 152ms/step - loss: 0.4394 - val_loss: 0.5600
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 95s 152ms/step - loss: 0.3000 - val_loss: 0.5807
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 100s 160ms/step - loss: 0.2348 - val_loss: 0.5720
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 136s 151ms/step - loss: 0.2012 - val_loss: 0.5947
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 95s 152ms/step - loss: 0.1595 - val_loss: 0.6383


In [26]:
## Save model file
model.save('simple_rnn_imdb.h5')